# In-Class Activity 6: Decision Trees, Random Forests, and Boosting

## Dataset: Landsat Satellite Land-Cover Classification

In this activity, you will use the **Statlog Landsat Satellite** dataset to compare several tree-based machine-learning models:

- A single decision tree
- Random forest
- AdaBoost
- XGBoost

Each row represents a small 3 by 3 pixel neighborhood from Landsat satellite imagery. The predictors are spectral measurements from those pixels, and the target is the land-cover class of the central pixel.

## Learning goals

By the end of this activity, you should be able to:

- Explain why decision trees make **axis-aligned splits**.
- Visualize the limitations of a single decision tree boundary.
- Compare a single tree with ensemble tree methods.
- Explain the difference between bagging and boosting.
- Use train/test splits and cross-validation appropriately.
- Tune tree-based models without leaking information from the test set.
- Interpret model behavior using feature importance and permutation importance.

## 1. Setup

Run this cell first. If `xgboost` is missing, install it with:

```bash
pip install xgboost
```

In [ ]:
import warnings

import matplotlib.pyplot as plt
from matplotlib.colors import BoundaryNorm, ListedColormap
import numpy as np
import pandas as pd
import seaborn as sns

from sklearn.datasets import fetch_openml
from sklearn.dummy import DummyClassifier
from sklearn.ensemble import AdaBoostClassifier, RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.inspection import permutation_importance
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder
from sklearn.tree import DecisionTreeClassifier, plot_tree

try:
    from xgboost import XGBClassifier
except ImportError as exc:
    raise ImportError("Please install xgboost before running this activity: pip install xgboost") from exc

warnings.filterwarnings("ignore", category=FutureWarning)

RANDOM_STATE = 42
sns.set_theme(style="whitegrid")

## 2. Load the Landsat Dataset

The dataset is available through OpenML and can be loaded directly with `scikit-learn`.

Original land-cover classes:

| Class | Land-cover label |
|---:|---|
| 1 | red soil |
| 2 | cotton crop |
| 3 | grey soil |
| 4 | damp grey soil |
| 5 | soil with vegetation stubble |
| 7 | very damp grey soil |

In [ ]:
landsat = fetch_openml(data_id=182, as_frame=True, parser="auto")

df = landsat.frame.copy()
original_target = landsat.target_names[0]

# Rename predictors so their structure is easier to understand.
# The 36 predictors represent 9 pixels times 4 spectral bands.
feature_cols_original = [col for col in df.columns if col != original_target]
feature_cols = [f"pixel_{pixel}_band_{band}" for pixel in range(1, 10) for band in range(1, 5)]
rename_map = dict(zip(feature_cols_original, feature_cols))

df = df.rename(columns=rename_map)
df["land_cover"] = df[original_target].astype(str)
df = df.drop(columns=[original_target])

df.head()

## 3. Initial Data Checks

Before modeling, check the number of rows, columns, missing values, and class balance.

In [ ]:
print("Rows, columns:", df.shape)
print("\nMissing values:")
print(df.isna().sum().sum())

class_counts = df["land_cover"].value_counts().sort_index()
class_counts

In [ ]:
class_labels = {
    "1.": "red soil",
    "2.": "cotton crop",
    "3.": "grey soil",
    "4.": "damp grey soil",
    "5.": "soil with vegetation stubble",
    "7.": "very damp grey soil",
}

class_counts_named = class_counts.rename(index=class_labels)

plt.figure(figsize=(8, 4))
sns.barplot(x=class_counts_named.index, y=class_counts_named.values, color="steelblue")
plt.xticks(rotation=30, ha="right")
plt.ylabel("Number of observations")
plt.title("Landsat Land-Cover Class Counts")
plt.tight_layout()
plt.show()

### Question 1

Is this classification problem balanced? Which land-cover classes are most and least common?

## 4. Why Decision Trees Have Axis-Aligned Splits

Decision trees split one feature at a time.

Examples:

```text
pixel_5_band_1 <= 66
pixel_5_band_2 <= 42
pixel_5_band_4 <= 87
```

That means a decision tree creates boundaries that are vertical or horizontal in a two-feature plot. These are called **axis-aligned splits**.

This is powerful and interpretable, but it can be inefficient when the real class boundary is diagonal, curved, or depends on a smooth combination of predictors.

## 5. Build a Two-Feature Dataset for Visualization

To visualize decision boundaries, we will use only two predictors from the central pixel:

- `pixel_5_band_1`
- `pixel_5_band_2`

This is intentionally limited. Later, we will use all 36 spectral predictors.

In [ ]:
two_feature_cols = ["pixel_5_band_1", "pixel_5_band_2"]

df_two = df[two_feature_cols + ["land_cover"]].dropna().copy()

X_two = df_two[two_feature_cols]
y_land_cover = df_two["land_cover"]

label_encoder = LabelEncoder()
y_two = label_encoder.fit_transform(y_land_cover)
class_codes = label_encoder.classes_
class_names = [class_labels[code] for code in class_codes]

# Use one shared color map for points and decision-boundary regions.
class_color_map = dict(zip(class_names, sns.color_palette("tab10", n_colors=len(class_names))))
background_cmap = ListedColormap([class_color_map[name] for name in class_names])
background_levels = np.arange(-0.5, len(class_names) + 0.5, 1)
background_norm = BoundaryNorm(background_levels, background_cmap.N)

X_two_train, X_two_test, y_two_train, y_two_test = train_test_split(
    X_two,
    y_two,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y_two,
)

class_names

In [ ]:
plot_df = df_two.copy()
plot_df["land_cover_name"] = plot_df["land_cover"].map(class_labels)

plt.figure(figsize=(8, 6))
sns.scatterplot(
    data=plot_df,
    x="pixel_5_band_1",
    y="pixel_5_band_2",
    hue="land_cover_name",
    s=35,
    alpha=0.75,
    edgecolor="none"
)
plt.title("Landsat Classes Using Two Central-Pixel Spectral Bands")
plt.legend(title="Land cover", bbox_to_anchor=(1.05, 1), loc="upper left")
plt.tight_layout()
plt.show()

### Question 2

Looking at the scatterplot, where might a single decision tree struggle? Do the classes appear separable by only horizontal and vertical cuts?

## 6. Fit a Shallow Decision Tree

A shallow tree is easy to interpret, but it may be too simple.

In [ ]:
shallow_tree = DecisionTreeClassifier(
    max_depth=2,
    random_state=RANDOM_STATE
)

shallow_tree.fit(X_two_train, y_two_train)

print("Training accuracy:", accuracy_score(y_two_train, shallow_tree.predict(X_two_train)))
print("Test accuracy:", accuracy_score(y_two_test, shallow_tree.predict(X_two_test)))

In [ ]:
plt.figure(figsize=(16, 7))
plot_tree(
    shallow_tree,
    feature_names=two_feature_cols,
    class_names=class_names,
    filled=True,
    rounded=True,
    fontsize=8,
)
plt.title("Shallow Decision Tree")
plt.show()

## 7. Decision Boundary Helper Function

The background color shows the model's predicted class across the two-feature space.

In [ ]:
def plot_decision_boundary(model, X_train, y_train, X_test, y_test, title):
    x_min, x_max = X_train.iloc[:, 0].min() - 5, X_train.iloc[:, 0].max() + 5
    y_min, y_max = X_train.iloc[:, 1].min() - 5, X_train.iloc[:, 1].max() + 5

    xx, yy = np.meshgrid(
        np.linspace(x_min, x_max, 350),
        np.linspace(y_min, y_max, 350)
    )

    grid = pd.DataFrame(
        np.c_[xx.ravel(), yy.ravel()],
        columns=X_train.columns
    )

    Z = model.predict(grid).reshape(xx.shape)

    plt.contourf(
        xx,
        yy,
        Z,
        levels=background_levels,
        cmap=background_cmap,
        norm=background_norm,
        alpha=0.25
    )

    train_df = X_train.copy()
    train_df["land_cover"] = label_encoder.inverse_transform(y_train)
    train_df["land_cover_name"] = train_df["land_cover"].map(class_labels)
    train_df["split"] = "Training"

    test_df = X_test.copy()
    test_df["land_cover"] = label_encoder.inverse_transform(y_test)
    test_df["land_cover_name"] = test_df["land_cover"].map(class_labels)
    test_df["split"] = "Test"

    point_df = pd.concat([train_df, test_df], ignore_index=True)

    sns.scatterplot(
        data=point_df,
        x=X_train.columns[0],
        y=X_train.columns[1],
        hue="land_cover_name",
        hue_order=class_names,
        palette=class_color_map,
        style="split",
        markers={"Training": "o", "Test": "X"},
        s=45,
        alpha=0.85,
        edgecolor="black",
        linewidth=0.3,
    )

    plt.title(title)
    plt.xlabel(X_train.columns[0])
    plt.ylabel(X_train.columns[1])
    plt.legend(title="Land cover / split", bbox_to_anchor=(1.05, 1), loc="upper left")
    plt.tight_layout()

In [ ]:
plt.figure(figsize=(8, 6))
plot_decision_boundary(
    shallow_tree,
    X_two_train,
    y_two_train,
    X_two_test,
    y_two_test,
    "Shallow Decision Tree: Axis-Aligned Splits"
)
plt.xlim(-2.5, 2.5)
plt.ylim(-2.5, 2.5)
plt.show()

### Question 3

What do you notice about the decision boundary? Why does it look like rectangles or stair steps?

## 8. Compare Single Tree Depths

Deeper trees are more flexible, but they can overfit.

In [ ]:
tree_depths = [1, 2, 5, None]
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.ravel()

for ax, depth in zip(axes, tree_depths):
    tree = DecisionTreeClassifier(max_depth=depth, random_state=RANDOM_STATE)
    tree.fit(X_two_train, y_two_train)

    plt.sca(ax)
    title = f"Decision Tree max_depth={depth}"
    plot_decision_boundary(tree, X_two_train, y_two_train, X_two_test, y_two_test, title)
    plt.xlim(-2.5, 2.5)
    plt.ylim(-2.5, 2.5)
plt.tight_layout()
plt.show()

### Question 4

How does the decision boundary change as the tree gets deeper? Which model seems most likely to overfit?

## 9. Compare Tree-Based Ensembles on the Same Two Features

Now compare:

- Decision tree
- Random forest
- AdaBoost
- XGBoost

This section is still using only two features so the boundaries can be visualized.

In [ ]:
two_feature_models = {
    "Decision tree": DecisionTreeClassifier(max_depth=5, random_state=RANDOM_STATE),
    "Random forest": RandomForestClassifier(
        n_estimators=200,
        max_depth=6,
        random_state=RANDOM_STATE
    ),
    "AdaBoost": AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE),
        n_estimators=100,
        learning_rate=0.5,
        random_state=RANDOM_STATE
    ),
    "XGBoost": XGBClassifier(
        n_estimators=100,
        max_depth=2,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.9,
        eval_metric="mlogloss",
        random_state=RANDOM_STATE
    ),
}

for model_name, model in two_feature_models.items():
    model.fit(X_two_train, y_two_train)
    train_acc = accuracy_score(y_two_train, model.predict(X_two_train))
    test_acc = accuracy_score(y_two_test, model.predict(X_two_test))
    print(f"{model_name:15s} train accuracy = {train_acc:.3f}; test accuracy = {test_acc:.3f}")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12))
axes = axes.ravel()

for ax, (model_name, model) in zip(axes, two_feature_models.items()):
    plt.sca(ax)
    plot_decision_boundary(
        model,
        X_two_train,
        y_two_train,
        X_two_test,
        y_two_test,
        model_name
    )
    plt.xlim(-2.5, 2.5)
    plt.ylim(-2.5, 2.5)
plt.tight_layout()
plt.show()

### Question 5

Compare the four decision boundaries.

1. Which model has the simplest boundary?
2. Which model has the most complex boundary?
3. Which model seems most likely to overfit?
4. Why do all tree-based boundaries still look somewhat blocky?

# Part 2: Best Practices With All Spectral Features

The two-feature plots are useful for learning, but they are not the best possible models. In real modeling, we should use a proper train/test split, cross-validation, preprocessing, and tuning.

We will now use all 36 spectral predictors.

Best-practice reminders:

- Keep the test set untouched until final evaluation.
- Use cross-validation for model comparison and tuning.
- Use pipelines to avoid preprocessing leakage.
- Tune complexity controls such as tree depth, number of trees, and learning rate.
- Compare against a simple baseline.

## 10. Prepare the Full Feature Dataset

In [ ]:
X = df[feature_cols]
y_labels = df["land_cover"]

y = label_encoder.fit_transform(y_labels)
class_codes = label_encoder.classes_
class_names = [class_labels[code] for code in class_codes]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.25,
    random_state=RANDOM_STATE,
    stratify=y,
)

print("Training rows:", X_train.shape[0])
print("Test rows:", X_test.shape[0])
print("Number of predictors:", X_train.shape[1])
print("Classes:", class_names)

## 11. Define Baseline and Candidate Models

Tree models do not require feature scaling. We include a median imputer in the pipeline as a best-practice safeguard, although this dataset should not have missing predictor values.

In [ ]:
models = {
    "Baseline": DummyClassifier(strategy="most_frequent"),
    "Decision tree": DecisionTreeClassifier(
        max_depth=8,
        min_samples_leaf=5,
        random_state=RANDOM_STATE
    ),
    "Random forest": RandomForestClassifier(
        n_estimators=300,
        max_features="sqrt",
        min_samples_leaf=2,
        random_state=RANDOM_STATE
    ),
    "AdaBoost": AdaBoostClassifier(
        estimator=DecisionTreeClassifier(max_depth=1, random_state=RANDOM_STATE),
        n_estimators=150,
        learning_rate=0.5,
        random_state=RANDOM_STATE
    ),
    "XGBoost": XGBClassifier(
        n_estimators=150,
        max_depth=2,
        learning_rate=0.05,
        subsample=0.9,
        random_state=RANDOM_STATE
    ),
}

pipelines = {
    name: Pipeline(
        steps=[
            ("imputer", SimpleImputer(strategy="median")),
            ("model", model)
        ]
    )
    for name, model in models.items()
}

## 12. Cross-Validation Comparison

Use cross-validation on the training set to compare models before touching the test set.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

cv_rows = []

for model_name, pipe in pipelines.items():
    scores = cross_validate(
        pipe,
        X_train,
        y_train,
        cv=cv,
        scoring=["accuracy", "f1_macro"],
        return_train_score=True,
        n_jobs=-1,
    )

    cv_rows.append({
        "model": model_name,
        "mean_train_accuracy": scores["train_accuracy"].mean(),
        "mean_cv_accuracy": scores["test_accuracy"].mean(),
        "mean_cv_f1_macro": scores["test_f1_macro"].mean(),
        "sd_cv_accuracy": scores["test_accuracy"].std(),
    })

cv_results = pd.DataFrame(cv_rows).sort_values("mean_cv_f1_macro", ascending=False)
cv_results

### Question 6

Which model has the best cross-validated macro F1 score? Does any model have much higher training accuracy than validation accuracy? What would that suggest?

## 13. Tune XGBoost

XGBoost has many tuning parameters. We will tune only a few so the activity stays manageable.

Important parameters:

- `max_depth`: tree complexity
- `learning_rate`: contribution of each tree
- `n_estimators`: number of boosting rounds
- `subsample`: fraction of rows used per tree

Best practice: tune on the training set using cross-validation, then evaluate once on the held-out test set.

In [ ]:
xgb_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("model", XGBClassifier(
            eval_metric="mlogloss",
            random_state=RANDOM_STATE
        ))
    ]
)

param_grid = {
    "model__n_estimators": [150, 300, 375],
    "model__max_depth": [2, 4, 5, 6],
    "model__learning_rate": [0.1, 0.2, 0.25],
    "model__subsample": [0.8, 1.0],
}

grid_search = GridSearchCV(
    estimator=xgb_pipeline,
    param_grid=param_grid,
    scoring="f1_macro",
    cv=cv,
    n_jobs=-1,
)

grid_search.fit(X_train, y_train)

print("Best CV score:", grid_search.best_score_)
print("Best parameters:")
print(grid_search.best_params_)

### Question 7

Which parameters were selected for XGBoost? Were the selected trees shallow or deep? Why does `model__subsample` do?

## 14. Final Test-Set Evaluation

Now evaluate the selected models on the test set.

The test set should be used only after model comparison and tuning decisions are complete.

In [ ]:
final_models = {
    "Decision tree": pipelines["Decision tree"],
    "Random forest": pipelines["Random forest"],
    "AdaBoost": pipelines["AdaBoost"],
    "Tuned XGBoost": grid_search.best_estimator_,
}

test_rows = []

for model_name, pipe in final_models.items():
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)

    test_rows.append({
        "model": model_name,
        "test_accuracy": accuracy_score(y_test, y_pred),
    })

test_results = pd.DataFrame(test_rows).sort_values("test_accuracy", ascending=False)
test_results

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
axes = axes.ravel()

for ax, (model_name, pipe) in zip(axes, final_models.items()):
    y_pred = pipe.predict(X_test)
    cm = confusion_matrix(y_test, y_pred)

    disp = ConfusionMatrixDisplay(
        confusion_matrix=cm,
        display_labels=class_names
    )
    disp.plot(ax=ax, cmap="Blues", colorbar=False, values_format="d")
    ax.set_title(model_name)
    ax.tick_params(axis="x", rotation=45)
    for label in ax.get_xticklabels():
        label.set_horizontalalignment("right")
        
plt.tight_layout()
plt.show()

In [ ]:
for model_name, pipe in final_models.items():
    print("=" * 70)
    print(model_name)
    print("=" * 70)
    y_pred = pipe.predict(X_test)
    print(classification_report(y_test, y_pred, target_names=class_names))

### Question 8

Which model performs best on the test set? Is that the same model that performed best during cross-validation? If not, why might that happen?

## 15. Feature Importance and Permutation Importance

Tree-based models often provide built-in feature importance, but those values can be biased. Permutation importance is usually easier to explain:

> A feature is important if model performance gets worse when that feature is randomly shuffled.

Here we will use permutation importance on the tuned XGBoost model.

In [ ]:
best_model = grid_search.best_estimator_
best_model.fit(X_train, y_train)

perm = permutation_importance(
    best_model,
    X_test,
    y_test,
    scoring="f1_macro",
    n_repeats=10,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

importance_df = pd.DataFrame({
    "feature": X_test.columns,
    "importance_mean": perm.importances_mean,
    "importance_sd": perm.importances_std,
}).sort_values("importance_mean", ascending=False)

importance_df.head(12)

In [ ]:
plt.figure(figsize=(8, 6))
sns.barplot(
    data=importance_df.head(12),
    x="importance_mean",
    y="feature",
    color="steelblue"
)
plt.xlabel("Decrease in macro F1 after permutation")
plt.ylabel("Feature")
plt.title("Permutation Importance: Tuned XGBoost")
plt.tight_layout()
plt.show()

## 16. Best-Practice Reflection

Answer these questions in a few sentences each.

1. Why is a single decision tree easy to interpret but prone to overfitting?
2. Why do decision tree boundaries look blocky in two dimensions?
3. How is a random forest different from a single tree?
4. How is boosting different from random forest bagging?
5. Why is permutation importance useful, and what are its limitations?
6. For environmental remote-sensing datasets, why might random train/test splits still be too optimistic?